In [0]:
import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

# 1. Gerar dados simulados de um e-commerce
produtos = [
    ("Portátil Pro", "Eletrónica", 1500.0), 
    ("Rato sem fios", "Eletrónica", 50.0),
    ("Cadeira Ergonómica", "Mobiliário", 200.0), 
    ("Secretária", "Mobiliário", 300.0),
    ("Teclado Mecânico", "Eletrónica", 100.0)
]

dados_brutos = []
data_inicial = datetime(2026, 1, 1)

for i in range(1, 1001): # Gerando 1.000 transações
    prod = random.choice(produtos)
    dados_brutos.append((
        i,                                                         # id_transacao
        data_inicial + timedelta(days=random.randint(0, 100)),     # data_venda
        random.randint(1, 50),                                     # id_cliente
        prod[0],                                                   # produto
        prod[1],                                                   # categoria
        random.randint(1, 5),                                      # quantidade
        prod[2],                                                   # valor_unitario
        random.choice(["CONCLUÍDO", "CONCLUÍDO", "CANCELADO", "PENDENTE"]) # status
    ))

# 2. Definir o Schema estrito para o PySpark
schema = StructType([
    StructField("id_transacao", IntegerType(), True),
    StructField("data_venda", DateType(), True),
    StructField("id_cliente", IntegerType(), True),
    StructField("produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("valor_unitario", DoubleType(), True),
    StructField("status", StringType(), True)
])

# 3. Criar o DataFrame Distribuído
df_bronze = spark.createDataFrame(dados_brutos, schema)

# 4. Guardar no Data Lake em formato Delta (A base do Lakehouse)
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("vendas_bronze")

# 5. Exibir os dados no ecrã
display(df_bronze)

id_transacao,data_venda,id_cliente,produto,categoria,quantidade,valor_unitario,status
1,2026-01-28,13,Portátil Pro,Eletrónica,1,1500.0,CONCLUÍDO
2,2026-04-10,38,Rato sem fios,Eletrónica,5,50.0,CANCELADO
3,2026-04-02,49,Portátil Pro,Eletrónica,3,1500.0,PENDENTE
4,2026-01-09,45,Secretária,Mobiliário,4,300.0,CONCLUÍDO
5,2026-03-07,17,Secretária,Mobiliário,4,300.0,CANCELADO
6,2026-01-04,43,Secretária,Mobiliário,2,300.0,CONCLUÍDO
7,2026-01-11,47,Portátil Pro,Eletrónica,4,1500.0,PENDENTE
8,2026-01-05,48,Teclado Mecânico,Eletrónica,1,100.0,CONCLUÍDO
9,2026-02-06,2,Teclado Mecânico,Eletrónica,4,100.0,CANCELADO
10,2026-02-05,50,Rato sem fios,Eletrónica,1,50.0,CONCLUÍDO


In [0]:
from pyspark.sql.functions import col, current_timestamp, upper

# 1. Ler os dados brutos da tabela Bronze
df_bronze = spark.read.table("vendas_bronze")

# 2. Aplicar as transformações da Camada Silver (Limpeza e Padronização)
df_silver = df_bronze.filter(col("status") == "CONCLUÍDO") \
                     .withColumn("categoria", upper(col("categoria"))) \
                     .withColumn("dt_processamento_silver", current_timestamp())

# 3. Guardar a nova versão no Data Lake em formato Delta
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("vendas_silver")

# 4. Exibir os dados limpos
print(f"Total de registos originais (Bronze): {df_bronze.count()}")
print(f"Total de registos válidos (Silver): {df_silver.count()}")
display(df_silver)

Total de registos originais (Bronze): 1000
Total de registos válidos (Silver): 489


id_transacao,data_venda,id_cliente,produto,categoria,quantidade,valor_unitario,status,dt_processamento_silver
1,2026-01-28,13,Portátil Pro,ELETRÓNICA,1,1500.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
4,2026-01-09,45,Secretária,MOBILIÁRIO,4,300.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
6,2026-01-04,43,Secretária,MOBILIÁRIO,2,300.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
8,2026-01-05,48,Teclado Mecânico,ELETRÓNICA,1,100.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
10,2026-02-05,50,Rato sem fios,ELETRÓNICA,1,50.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
12,2026-03-17,15,Secretária,MOBILIÁRIO,5,300.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
13,2026-02-17,36,Secretária,MOBILIÁRIO,1,300.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
14,2026-01-13,34,Secretária,MOBILIÁRIO,1,300.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
17,2026-03-22,43,Rato sem fios,ELETRÓNICA,3,50.0,CONCLUÍDO,2026-10-09T17:20:49.325Z
18,2026-01-02,21,Portátil Pro,ELETRÓNICA,5,1500.0,CONCLUÍDO,2026-10-09T17:20:49.325Z


In [0]:
from pyspark.sql.functions import col, sum as _sum, round as _round

# 1. Ler os dados limpos e validados da tabela Silver
df_silver = spark.read.table("vendas_silver")

# 2. Construir a Camada Gold (Métricas de Negócio)
# Multiplicamos quantidade pelo valor unitário e agrupamos por categoria e produto
df_gold = df_silver.withColumn("receita_transacao", col("quantidade") * col("valor_unitario")) \
                   .groupBy("categoria", "produto") \
                   .agg(
                       _round(_sum("receita_transacao"), 2).alias("faturamento_total"),
                       _sum("quantidade").alias("volume_vendas")
                   ) \
                   .orderBy(col("faturamento_total").desc())

# 3. Guardar a tabela final no Data Lake em formato Delta
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("vendas_gold")

# 4. Exibir o resultado (O "Dashboard" Executivo)
display(df_gold)

categoria,produto,faturamento_total,volume_vendas
ELETRÓNICA,Portátil Pro,526500.0,351
MOBILIÁRIO,Secretária,73800.0,246
MOBILIÁRIO,Cadeira Ergonómica,60200.0,301
ELETRÓNICA,Teclado Mecânico,31400.0,314
ELETRÓNICA,Rato sem fios,13950.0,279
